[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/45_full_grpo_loss_solution.ipynb)

# 🔴 Solution: Full GRPO Loss

Reference solution for a clipped GRPO objective with a reference KL penalty.


In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
from torch import Tensor


In [ ]:
# ✅ SOLUTION

def full_grpo_loss(logps: Tensor, old_logps: Tensor, ref_logps: Tensor,
                   rewards: Tensor, group_ids: Tensor, completion_mask: Tensor,
                   clip_ratio: float = 0.2, beta: float = 0.1, eps: float = 1e-5) -> Tensor:
    unique_ids = group_ids.unique()
    advantages = torch.empty_like(rewards)
    for gid in unique_ids:
        mask = group_ids == gid
        r_g = rewards[mask]
        mean_g = r_g.mean()
        std_g = r_g.std(unbiased=False)
        advantages[mask] = (r_g - mean_g) / (std_g + eps)

    advantages = advantages.detach().unsqueeze(1)
    old_logps = old_logps.detach()
    ref_logps = ref_logps.detach()
    valid = completion_mask > 0

    ratio = torch.ones_like(logps)
    ratio[valid] = torch.exp(logps[valid] - old_logps[valid])
    unclipped = ratio * advantages
    clipped = torch.clamp(ratio, 1.0 - clip_ratio, 1.0 + clip_ratio) * advantages

    # Schulman-style positive KL estimator used in many GRPO implementations.
    kl = torch.zeros_like(logps)
    delta = ref_logps[valid] - logps[valid]
    kl[valid] = torch.exp(delta) - delta - 1.0

    token_objective = torch.minimum(unclipped, clipped) - beta * kl
    token_counts = completion_mask.sum(dim=-1).clamp_min(1.0)
    seq_objective = (token_objective * completion_mask).sum(dim=-1) / token_counts
    return -seq_objective.mean()


In [ ]:
# Verify
logps = torch.tensor([[-0.2, -0.1, -0.3], [-0.6, -0.5, -0.4], [-0.3, -0.8, -1.0], [-0.9, -1.1, -1.2]])
old_logps = torch.tensor([[-0.3, -0.2, -0.4], [-0.4, -0.4, -0.4], [-0.5, -0.6, -0.9], [-0.7, -1.0, -1.3]])
ref_logps = torch.tensor([[-0.25, -0.15, -0.35], [-0.55, -0.45, -0.45], [-0.35, -0.7, -0.95], [-0.8, -1.0, -1.1]])
rewards = torch.tensor([1.0, 0.8, 0.3, 0.1])
group_ids = torch.tensor([0, 0, 1, 1])
completion_mask = torch.tensor([[1, 1, 1], [1, 1, 0], [1, 1, 1], [1, 0, 0]], dtype=torch.float32)
print('Loss:', full_grpo_loss(logps, old_logps, ref_logps, rewards, group_ids, completion_mask))


In [ ]:
# Run judge
from torch_judge import check
check('full_grpo_loss')
